# LangChain Basics

In [1]:
!pip install -q langchain langchain-openai langchain-core openai tiktoken cohere


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)

This is taking too long, we give up.



In [3]:
import sys; print(sys.executable)

e:\anaconda3\envs\MY_RAG\python.exe


In [5]:
from langchain_classic.document_loaders import UnstructuredPDFLoader

In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

In [25]:
# Core imports for the notebook

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage,SystemMessage

In [36]:
# This is the simplest LangChain usage:
# create a model, send a message, get a response.

llm = ChatOpenAI(model="gpt-5-nano", temperature=0.2 , api_key= openai_api_key, reasoning_effort="minimal")

llm.invoke("explain global warming")

AIMessage(content="Global warming is the long-term increase in the average temperature of Earth's surface and lower atmosphere. It’s driven mainly by human activities and reinforced by natural factors. Here’s a clear overview:\n\nWhat’s happening\n- Since the late 19th century, the average global surface temperature has risen by about 1.2°C (roughly 2.2°F), with the warmest decades on record in the past 20–30 years.\n- Heat is building up in the climate system, especially in the atmosphere and the oceans. The oceans have absorbed the majority of this excess heat, which also drives other changes like sea-level rise and stronger storms.\n\nWhy it’s happening\n- Greenhouse gases (GHGs): Humans release gases such as carbon dioxide (CO2), methane (CH4), nitrous oxide (N2O), and fluorinated gases. These trap heat in the lower atmosphere and the surface, creating a warming effect.\n- Deforestation and land-use changes: Removing forests reduces the land’s capacity to absorb CO2, leaving more i

In [37]:
# Prompt templates let you reuse the same structure with different inputs.

prompt = ChatPromptTemplate.from_messages(messages=[
    ("system", "You are a helpful AI ASSISTANT"),
    ("user", "Explain {topic} in beginner-friendly words")
])


formatted_prompt = prompt.invoke({"topic":"Object Oriented Programming"})
formatted_prompt


ChatPromptValue(messages=[SystemMessage(content='You are a helpful AI ASSISTANT', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain Object Oriented Programming in beginner-friendly words', additional_kwargs={}, response_metadata={})])

In [38]:
chain = prompt | llm

chain.invoke({"topic":"Langchain"})

AIMessage(content='Sure! Here’s a beginner-friendly explanation.\n\nWhat Langchain is (in simple terms)\n- Langchain is a library (a set of pre-made tools) that helps you build programs that use large language models (like GPT-4) more easily.\n- It gives you ready-made building blocks to connect a language model to data, tools, and workflows.\n\nWhy people use Langchain\n- You want to do smart things with a language model, like:\n  - Answering questions using your own knowledge base\n  - Writing summaries or reports from documents\n  - Turning natural language instructions into automated tasks\n  - Building chatbots that can do things beyond just chatting\n\nKey ideas and components (in plain language)\n- Prompt templates: Reusable “fill-in-the-blank” prompts. You specify a pattern, and Langchain fills in the blanks with your data.\n- Chains: A sequence of steps where the output of one step becomes the input for the next. It’s like a tiny workflow.\n  - Example: Read a document → Summa

In [45]:
from pydantic import BaseModel, Field

class StudyPlan(BaseModel) :
    topic: str = Field(description="Topic to study")
    level: str = Field(description="Beginner, Intermediate or Advance")
    steps: list[str] = Field (description="Learning Steps")

structured_llm = llm.with_structured_output(StudyPlan)

plan = structured_llm.invoke("Create a begginer study plan for Langchain in 5 Steps")

print(plan)


topic='LangChain' level='Beginner' steps=['Understand the basics: what LangChain is, how it helps build AI apps with language models, and its common components (LLMs, prompts, chains, memory, and agents). Install LangChain and set up a Python environment.', 'Learn core concepts: prompts and prompt templates, chains (simple to complex), memory (conversation history), and tools/agents. Create small examples: a basic prompt, a simple chain that calls an LLM, and a memory-enabled chat.', 'Build simple applications: create a basic chat bot using a chain, implement a search tool to fetch info, and integrate a QA chain that answers questions from a document. Practice with OpenAI or local models, and handle API keys securely.', 'Work with tools, agents, and memory: add tools (e.g., search, calculator), build an agent that decides which tool to use, and implement memory to maintain context across turns. Experiment with different memory strategies (short-term, long-term).', 'Projects and best pr

list

In [44]:
for i in plan.steps:
    print(i)



Understand what Langchain is and why it's useful for building LLM-powered apps; familiarize yourself with core concepts like chains, prompts, and agents.
Install Langchain in your development environment and run a simple example that calls an LLM to generate a response from a prompt.
Explore basic building blocks: prompts design, prompt templates, and simple chains (e.g., LLM chain, Sequential/Simple chain) to process user input and produce outputs.
Learn about agents and tools: understand how to create an agent that can decide actions (like calling a tool or API) based on a user request; implement a basic tool (e.g., a calculator or search tool) and integrate with an agent.
Build a small end-to-end project: create a chat assistant that uses prompts, a few chained steps, and an agent with at least one tool; test, iterate on prompt design, handle errors, and review best practices for reliability and safety.


In [47]:
def get_weather(city: str) -> str:
    return f"The weather in {city} is sunny and warm."
print(get_weather("karachi"))

The weather in karachi is sunny and warm.


In [74]:
#converting functions to tools


from langchain_core.tools import tool

@tool
def get_time_in_city(city: str) -> str:
    """Get the Current time in a city"""
    return f"Current time in {city} is 10:00 AM"

print(get_time_in_city.description)


Get the Current time in a city


In [76]:
# Modern LangChain emphasizes create_agent.
# The docs show create_agent as the basic way to build an agent with tools.

from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[get_time_in_city],
    system_prompt="You are a helpful assistant."
)

result = agent.invoke({
    "messages": [
        {"role": "user",
         "content": "Use the tool to tell me the time info for Hyderabad."}
    ]
})

print(result["messages"][-1].content)

The current time in Hyderabad is 10:00 AM.


In [77]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

memory_agent = create_agent(
    model = "openai:gpt-5-nano",
    tools=[get_time_in_city],
    system_prompt="You are a helpful assistant",
    checkpointer=checkpointer
)

config = {"configurable":{"thread_id":"demo-thread"}}

r1 = memory_agent.invoke(
    {"messages" : [ {"role" : "user",
                     "content":"My name is Muneeb"}]},
    config=config
)

r2 = memory_agent.invoke(
    {"messages" : [{"role":"user",
                    "content":"what is my name ? "}]},
    config=config
)

print(r2["messages"][-1].content)

Your name is Muneeb. Would you like me to address you as Muneeb in our chats?


# BASIC RAG LANGCHAIN

In [ ]:
from 